In [2]:
import subprocess

result = subprocess.run(
    ["python", "-m", "src.retinocare.models.train",
     "--config", "configs/train_config.yaml",
     "--model", "baseline_cnn"],
    cwd="..", capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr)

Using device: cpu
Epoch 1/30 | train_loss=1.4816 train_f1=0.5156 | val_loss=1.3711 val_f1=0.6034
  -> new best val_f1=0.6034, saved to models\baseline_cnn.pt
Epoch 2/30 | train_loss=1.3547 train_f1=0.5708 | val_loss=1.3362 val_f1=0.5880
Epoch 3/30 | train_loss=1.3028 train_f1=0.5908 | val_loss=1.2877 val_f1=0.5200
Epoch 4/30 | train_loss=1.2698 train_f1=0.6217 | val_loss=1.3091 val_f1=0.6154
  -> new best val_f1=0.6154, saved to models\baseline_cnn.pt
Epoch 5/30 | train_loss=1.2454 train_f1=0.6129 | val_loss=1.2477 val_f1=0.6755
  -> new best val_f1=0.6755, saved to models\baseline_cnn.pt
Epoch 6/30 | train_loss=1.2318 train_f1=0.6242 | val_loss=1.5712 val_f1=0.5783
Epoch 7/30 | train_loss=1.2257 train_f1=0.6102 | val_loss=1.2685 val_f1=0.6471
Epoch 8/30 | train_loss=1.1946 train_f1=0.6391 | val_loss=1.2812 val_f1=0.5049
Epoch 9/30 | train_loss=1.1683 train_f1=0.6122 | val_loss=1.2682 val_f1=0.6882
  -> new best val_f1=0.6882, saved to models\baseline_cnn.pt
Epoch 10/30 | train_loss=1.

In [10]:
import sys
sys.path.append("..")

import torch, yaml
from src.retinocare.data.dataset import get_dataloaders
from src.retinocare.models.baseline_cnn import BaselineCNN
from src.retinocare.evaluation.metrics import compute_classification_report, plot_confusion_matrix

with open("../configs/train_config.yaml") as f:
    config = yaml.safe_load(f)


config["data"]["train_csv"] = "../" + config["data"]["train_csv"]
config["data"]["val_csv"] = "../" + config["data"]["val_csv"]
config["data"]["test_csv"] = "../" + config["data"]["test_csv"]
config["data"]["image_dir"] = "../" + config["data"]["image_dir"]

_, _, test_dl = get_dataloaders(config)

model = BaselineCNN(num_classes=5)
model.load_state_dict(torch.load("../models/baseline_cnn.pt", map_location="cpu"))
model.eval()

all_preds, all_labels = [], []
with torch.no_grad():
    for images, labels in test_dl:
        logits = model(images)
        all_preds.extend(logits.argmax(dim=1).tolist())
        all_labels.extend(labels.tolist())

SEVERITY_LABELS = ["No DR", "Mild", "Moderate", "Severe", "Proliferative DR"]
report = compute_classification_report(all_labels, all_preds, SEVERITY_LABELS)
print("Weighted F1:", report["weighted avg"]["f1-score"])
print("Macro F1:", report["macro avg"]["f1-score"])

plot_confusion_matrix(all_labels, all_preds, SEVERITY_LABELS, "../docs/baseline_cnn_confusion_matrix.png")
print("Confusion matrix saved to docs/baseline_cnn_confusion_matrix.png")

Weighted F1: 0.6907100926461087
Macro F1: 0.47797117304823844
Confusion matrix saved to docs/baseline_cnn_confusion_matrix.png
